In [ ]:
import pandas as pd
import numpy as np
import statistics

from pathlib import Path
from startorch import profile

# Every path comes from project-config.toml through startorch.utils.paths.
msmarco = profile("msmarco")
full_en = profile("full-en")
print(msmarco)
print(full_en)

In [ ]:
collection = pd.read_csv(
    msmarco.collection,
    sep='\t',
    header=None,
    names=["pid", "passage"],
)
print(collection.head())

In [ ]:
queries = pd.read_csv(
    msmarco.query_dir / "queries.dev.small.tsv",
    sep='\t',
    header=None,
    names=["qid", "query"]
)
print(queries.head())
queries.shape

In [ ]:
qrels = pd.read_csv(
    msmarco.query_dir / "qrels.dev.small.tsv",
    sep='\t',
    header=None,
    names=["qid", "unused", "pid", "relevance"]
)
print(qrels.head(10))

In [ ]:
query_result_dir = msmarco.result_dir
print (query_result_dir)
perf_path = query_result_dir / "perf_raw.parquet"

perf = pd.read_parquet(
    perf_path,
)
perf = perf.drop(perf[perf["run_name"] == "queries.dev.tsv"].index)

print(perf.columns)
perf["max_rss"] = perf["max_rss"].astype(float).apply(lambda x: x/1024)
perf["query_latency_mean"] = perf['query_latency'].apply(lambda x: sum(x) / len(x) if len(x) > 0 else 0)
perf["query_latency_median"] = perf['query_latency'].apply(statistics.median)
perf["query_latency_p95"] = perf['query_latency'].apply(lambda x: np.percentile(x,95))
perf["query_latency_p99"] = perf['query_latency'].apply(lambda x: np.percentile(x,99))
perf["query_latency_max"] = perf['query_latency'].apply(lambda x: np.max(x))
perf["query_latency_min"] = perf['query_latency'].apply(lambda x: np.min(x))
df = perf[["run_name", "query_latency_mean", "query_latency_median", "query_latency_p95", "query_latency_p99", "query_latency_max", "query_latency_min", "engine_latency", "max_rss"]]
print(df)

In [ ]:
full_meta_path = full_en.meta_path
query_data_dir = full_en.query_dir

random = pd.read_csv(query_data_dir / "random_set.tsv", sep="\t", header=None)
common = pd.read_csv(query_data_dir / "common_set.tsv", sep="\t", header=None)
very_common = pd.read_csv(query_data_dir / "very_common_set.tsv", sep="\t",header=None)
rare = pd.read_csv(query_data_dir / "rare_set.tsv", sep="\t",header=None)
skewed = pd.read_csv(query_data_dir / "skewed_set.tsv", sep="\t",header=None)

In [ ]:
print(random.head())
print(random.shape)


In [ ]:
print(common.head())

In [ ]:
print(very_common.head())

In [ ]:
print(rare.head())

In [ ]:
print(skewed.head())

In [ ]:
pd.set_option('display.max_columns', None)
query_result_dir = full_en.result_dir
print(query_result_dir)
perf_path = query_result_dir / "perf_raw.parquet"

perf = pd.read_parquet(
    perf_path,
)

print(perf.columns)
perf["max_rss"] = perf["max_rss"].astype(float).apply(lambda x: x/1024)
perf["query_latency_mean"] = perf['query_latency'].apply(lambda x: sum(x) / len(x) if len(x) > 0 else 0)
perf["query_latency_median"] = perf['query_latency'].apply(statistics.median)
perf["query_latency_p95"] = perf['query_latency'].apply(lambda x: np.percentile(x,95))
perf["query_latency_p99"] = perf['query_latency'].apply(lambda x: np.percentile(x,99))
perf["query_latency_max"] = perf['query_latency'].apply(lambda x: np.max(x))
perf["query_latency_min"] = perf['query_latency'].apply(lambda x: np.min(x))
df = perf[[
    "run_name", "query_latency_mean", "query_latency_median", "query_latency_p95", "query_latency_p99",
    "query_latency_max", "query_latency_min", "max_rss"]]
print(df)
mx = perf["engine_latency"].max()
mn = perf["engine_latency"].min()
print(mx, mn)

In [ ]:
import matplotlib.pyplot as plt

bmw_latency = np.sort(perf.loc[perf["run_name"] == "random_set.tsv_10", "query_latency"].iloc[0])
exhaustive_latency = np.sort(perf.loc[perf["run_name"] == "random_set.tsv_10_exhaustive", "query_latency"].iloc[0])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(bmw_latency) + 1), bmw_latency, label="BMW", linewidth=1.5)
ax.plot(np.arange(1, len(exhaustive_latency) + 1), exhaustive_latency, label="Exhaustive", linewidth=1.5)
ax.set_yscale("log")
ax.set_xlabel("Query rank (sorted ascending by latency)")
ax.set_ylabel("Query latency (ms, log scale)")
ax.set_title("Sorted query latency — random_set (k=10): BMW vs. Exhaustive")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
bmw_latency = np.sort(perf.loc[perf["run_name"] == "common_set.tsv_1000", "query_latency"].iloc[0])
exhaustive_latency = np.sort(perf.loc[perf["run_name"] == "common_set.tsv_1000_exhaustive", "query_latency"].iloc[0])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(bmw_latency) + 1), bmw_latency, label="BMW", linewidth=1.5)
ax.plot(np.arange(1, len(exhaustive_latency) + 1), exhaustive_latency, label="Exhaustive", linewidth=1.5)
ax.set_yscale("log")
ax.set_xlabel("Query rank (sorted ascending by latency)")
ax.set_ylabel("Query latency (ms, log scale)")
ax.set_title("Sorted query latency — common_set (k=1000): BMW vs. Exhaustive")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
bmw_latency = np.sort(perf.loc[perf["run_name"] == "very_common_set.tsv_10", "query_latency"].iloc[0])[::20]
exhaustive_latency = np.sort(perf.loc[perf["run_name"] == "very_common_set.tsv_10_exhaustive", "query_latency"].iloc[0])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(bmw_latency) + 1), bmw_latency, label="BMW", linewidth=1.5)
ax.plot(np.arange(1, len(exhaustive_latency) + 1), exhaustive_latency, label="Exhaustive", linewidth=1.5)
ax.set_yscale("log")
ax.set_xlabel("Query rank (sorted ascending by latency)")
ax.set_ylabel("Query latency (ms, log scale)")
ax.set_title("Sorted query latency — very_common_set (k=10): BMW vs. Exhaustive")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
pd.set_option('display.max_columns', None)
query_result_dir = full_en.result_dir
print(query_result_dir)
correctness_path = query_result_dir / "correctness_raw.parquet"

correctness = pd.read_parquet(
    correctness_path,
)

bmw_min = pd.to_numeric(correctness["bmw_min_score"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
exhaustive_min = pd.to_numeric(correctness["exhaustive_min_score"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)

bmw_max = pd.to_numeric(correctness["bmw_max_score"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
exhaustive_max = pd.to_numeric(correctness["exhaustive_max_score"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)

TOL = 1e-6

correctness["min_matches"] = np.isclose(bmw_min, exhaustive_min, rtol=0, atol=TOL, equal_nan=True)
correctness["max_matches"] = np.isclose(bmw_max, exhaustive_max, rtol=0, atol=TOL, equal_nan=True)

correctness["sum_missing"] = correctness["missing_docids"].apply(lambda per_query: sum(len(q) for q in per_query))
correctness["sum_extra"] = correctness["extra_docids"].apply(lambda per_query: sum(len(q) for q in per_query))

# correctness["sum_missing"] = correctness["missing_docids"].str.len().sum()
# correctness["sum_extra"] = correctness["extra_docids"].str.len().sum()

print(correctness.head())

print(correctness.columns)
print(correctness[["run_name", "query_matches", "min_matches", "max_matches", "sum_missing", "sum_extra"]])